# Statement Review — real-document walkthrough

Runs each stage of `api/statement_review/` against a real PDF so you can inspect
the intermediate output of every step:

1. **parse** — pypdf text extraction (local, no network)
2. **redact** — regex PII scrub (local; this is the privacy boundary)
3. **extract** — OpenAI structured output → transactions *(first LLM call)*
4. **review** — flags avoidable spend using your app context *(second LLM call)*
5. **crosscheck** — pure-Python match against your `user_expenses` rows

**Privacy note:** the raw (un-redacted) statement text is deliberately never
printed in this notebook — only stats about it. Cell outputs get saved into the
`.ipynb` file, and this repo is under git. Only the redacted text is displayed.

**How to run:** from the repo root —
```bash
uv run --with jupyter jupyter lab notebooks/test_statement_review.ipynb
```
Requires `OPENAI_API_KEY` in `.env` for stages 3–4.

In [ ]:
# --- Setup & config -------------------------------------------------------
import os, re, sys
from pathlib import Path

REPO_ROOT = Path.cwd() if (Path.cwd() / "api").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

PDF_PATH = Path("/home/aswinmanohar/Downloads/Konto_1935718393-Auszug_2026_0006 (1).PDF")
STATEMENT_TYPE = "bank"          # "bank" | "credit_card"
NAMES_TO_REDACT = []              # e.g. ["Aswin Manohar"] — the app sends your Google full_name automatically

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.1")
USER_ID = os.getenv("PERSONAL_USER_ID")

assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"
print(f"PDF:      {PDF_PATH.name} ({PDF_PATH.stat().st_size:,} bytes)")
print(f"model:    {MODEL}")
print(f"user_id:  {'set' if USER_ID else 'MISSING (set PERSONAL_USER_ID in .env)'}")
print(f"api key:  {'set' if os.getenv('OPENAI_API_KEY') else 'MISSING — stages 3-4 will fail until OPENAI_API_KEY is in .env'}")

pdf_bytes = PDF_PATH.read_bytes()

## Stage 1 — parse (pypdf, local)
Stats only; raw text is not printed (it still contains PII).

In [ ]:
from api.statement_review.parser import extract_text

raw_text = extract_text(pdf_bytes)
lines = raw_text.splitlines()
print(f"characters: {len(raw_text):,}")
print(f"lines:      {len(lines):,}")
print(f"non-empty:  {sum(1 for l in lines if l.strip()):,}")

## Stage 2 — redact (local regex, the privacy boundary)
Only the text this cell produces is ever allowed to reach OpenAI.
Masks: balance lines, address lines (street+number and postal-code+city
header/footer lines; merchant streets inside transaction lines are kept),
IBANs, phone numbers, card/account numbers, partial cards (`**** 1234`),
account-number labels, bare unlabeled 8+ digit numbers (Konto-Nr / BLZ /
reference numbers), and any names you list.

In [ ]:
from api.statement_review.redactor import redact

env_names = [n.strip() for n in os.getenv("REDACT_NAMES", "").split(",") if n.strip()]
merged_names = list(dict.fromkeys(env_names + NAMES_TO_REDACT))

redaction = redact(raw_text, extra_names=merged_names)
redacted_text = redaction.text

print("masked_counts:", redaction.masked_counts or "(nothing matched!)")
print("names list:   ", merged_names or "(none — add yours to NAMES_TO_REDACT)")
print("=" * 70)
print(redacted_text[:3000])
print("..." if len(redacted_text) > 3000 else "")

In [ ]:
# Leak check: long digit runs that survived redaction. Dates/amounts are short,
# so any 10+ digit run here deserves a look before trusting the scrub.
survivors = re.findall(r"(?:\d[ \-/]?){9,}\d", redacted_text)
if survivors:
    print(f"{len(survivors)} long digit run(s) survived — inspect these:")
    for s in sorted(set(survivors))[:20]:
        print("  ", s)
else:
    print("OK — no 10+ digit runs left in the redacted text.")

## Stage 3 — extract transactions (LLM call 1)
Structured output with validation + up to 2 retries; reconciles against the statement's own printed total when present.

In [ ]:
from openai import OpenAI
from api.statement_review.extractor import extract_transactions

client = OpenAI()  # reads OPENAI_API_KEY
extraction = extract_transactions(redacted_text, STATEMENT_TYPE, client, MODEL)
txns = extraction.transactions

print(f"{len(txns)} transactions extracted\n")
print(f"{'date':<12}{'direction':<8}{'amount':>10}  {'category':<14}description")
print("-" * 78)
for t in txns:
    print(f"{t.date:<12}{t.direction:<8}{t.amount:>10.2f}  {t.category:<14}{t.description[:38]}")

debits = [t for t in txns if t.direction == "debit"]
credits = [t for t in txns if t.direction == "credit"]
print("-" * 78)
print(f"debits:  {len(debits):>3}  total {sum(t.amount for t in debits):>10.2f}")
print(f"credits: {len(credits):>3}  total {sum(t.amount for t in credits):>10.2f}")
if extraction.total_debits is not None:
    print(f"statement's own printed debit total: {extraction.total_debits:.2f}")

## Stage 4 — review flags (LLM call 2)
Builds context from Supabase (income + 3-month category baseline from `user_expenses`, tombstones excluded) and asks the model to flag only clearly avoidable spend.

In [ ]:
from api.dependencies import get_supabase_client
from api.statement_review.reviewer import build_context, review_transactions

supabase = get_supabase_client()
assert supabase is not None, "Supabase client not configured (SUPABASE_URL / SUPABASE_KEY)"
assert USER_ID, "PERSONAL_USER_ID missing from .env"

context = build_context(supabase, USER_ID)
print(f"monthly income:    {context.monthly_income:.2f}")
print(f"recurring bills:   {len(context.recurring)}")
print(f"category baseline: {context.baseline}")
print()

flags = review_transactions(txns, context, client, MODEL)
if not flags:
    print("No flags — the model found nothing clearly avoidable.")
for f in sorted(flags, key=lambda f: {"high": 0, "medium": 1, "low": 2}.get(f.severity, 3)):
    t = f.transaction
    print(f"[{f.severity.upper():<6}] {f.flag_type:<22} {t.date}  {t.amount:>8.2f}  {t.description[:32]}")
    print(f"         {f.reason}  (est. monthly saving: {f.monthly_saving_estimate:.2f})")
print()
print(f"total potential monthly saving: {sum(f.monthly_saving_estimate for f in flags):.2f}")

## Stage 5 — crosscheck vs your app expenses (no LLM)
Match rule: amount within ±0.01 AND date within ±3 days; name similarity as tie-breaker.

In [ ]:
from api.statement_review.crosscheck import crosscheck

rows = (
    supabase.table("user_expenses")
    .select("id, name, amount, vendor, created_at")
    .eq("user_key", USER_ID)
    .eq("deleted", False)
    .execute()
).data or []
print(f"{len(rows)} live expense rows in the app\n")

check = crosscheck(txns, rows)

print(f"missing_in_app ({len(check.missing_in_app)}): on the statement, not in the app")
for t in check.missing_in_app:
    print(f"   {t.date}  {t.amount:>8.2f}  {t.description[:44]}")

print(f"\nmissing_on_statement ({len(check.missing_on_statement)}): in the app, not on the statement")
for r in check.missing_on_statement:
    print(f"   {str(r.get('created_at'))[:10]}  {float(r.get('amount') or 0):>8.2f}  {r.get('name')}")

print(f"\namount_mismatch ({len(check.amount_mismatch)}):")
for m in check.amount_mismatch:
    print(f"   {m.statement_tx.description[:30]}  statement {m.statement_tx.amount:.2f} vs app {float(m.app_expense.get('amount') or 0):.2f}  (delta {m.delta:+.2f})")

## End-to-end — `run_review()` exactly as the API endpoint calls it

In [ ]:
from api.statement_review.pipeline import run_review

report = run_review(
    pdf_bytes,
    redact_enabled=True,
    statement_type=STATEMENT_TYPE,
    user_id=USER_ID,
    supabase=supabase,
    client=client,
    model=MODEL,
    extra_names=NAMES_TO_REDACT,
)

t = report.totals
print(f"statement_spend: {t.statement_spend:.2f}")
print(f"flagged_spend:   {t.flagged_spend:.2f}")
print(f"coverage_pct:    {t.coverage_pct:.1f}%  (share of statement debits already tracked in the app)")
print(f"flags:           {len(report.flags)}")
print(f"missing_in_app:  {len(report.crosscheck.missing_in_app)}")
print(f"redaction:       {report.redaction_preview.masked_counts}")